# 01 — Data pipeline
Download RTLCoder + MG-Verilog, download eval sets (eval-only, never trained on), build the corpus (normalise -> dedup -> contamination check -> tag -> split).

In [1]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess, shutil
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

# artifacts/corpus.jsonl and data/eval/*.jsonl are built once in this
# notebook but land on this run's local (ephemeral) VM disk -- another
# notebook tab is not guaranteed to reuse the same VM, so without this
# they'd be missing there (a FileNotFoundError: 'artifacts/corpus.jsonl'
# failure mode). This notebook copies them to this same Drive folder after
# building them (see the last cell below); restore them here if this VM
# doesn't have them locally yet -- e.g. you're re-running just the eval
# or corpus-build cell after a runtime restart. No-ops harmlessly if the
# local copy already exists or Drive doesn't have one yet.
DATA = '/content/drive/MyDrive/verilog-slm/data'
os.makedirs(DATA, exist_ok=True)
os.makedirs('data/eval', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
for rel_path in ['artifacts/corpus.jsonl', 'data/eval/verilogeval_v2.jsonl', 'data/eval/rtllm_v2.jsonl']:
    drive_path = f"{DATA}/{os.path.basename(rel_path)}"
    if os.path.exists(drive_path) and not os.path.exists(rel_path):
        shutil.copy(drive_path, rel_path)
        print(f"restored {rel_path} from Drive")

Mounted at /content/drive
Cloning into '/content/verilog-slm'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 133 (delta 55), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 128.57 KiB | 5.84 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/verilog-slm
Already up to date.


In [2]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 5.0 MB/s eta 0:00:00


In [3]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

torch: 2.11.0+cu128 | CUDA: 12.8 | GPU available: True


In [4]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
resolved verible URL: https://github.com/chipsalliance/verible/releases/download/v0.0-4200-g4fa5630e/verible-v0.0-4200-g4fa5630e-linux-static-x86_64.tar.gz
Icarus Verilog version 12.0 (stable) ()
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivlpp -V"
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivl -V -C"/tmp/ivrlhab28d4d" -C"/usr/lib/x86_64-linux-gnu/ivl/vvp.conf""
Yosys 0.33 (git sha1 2584903a060)
Version	v0.0-4200-g4fa5630e
Commit-Timestamp	2026-09-11T12:11:26Z
Built	2026-09-11T13:37:21Z


In [5]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-6ef47414-555d-bab3-a264-28b6e853adf2)


In [6]:
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)

# RTLCoder: "ishorn5/RTLCoder-v1.1" on the HF Hub is a MODEL repo, not a
# dataset -- load_dataset() on it 404s. The actual 27k instruction/code
# pairs ship as a JSONL file inside the RTLCoder GitHub repo itself
# (verified against the live repo before writing this): each line is
# {"Instruction": str, "Response": [str]} -- note Response is a
# single-element LIST, not a plain string.
import json, re, urllib.request

RTLCODER_URL = "https://raw.githubusercontent.com/DevinShang/RTLCoder/main/dataset/Resyn27k.json"
urllib.request.urlretrieve(RTLCODER_URL, "data/raw/_resyn27k.jsonl")

with open("data/raw/_resyn27k.jsonl") as fin, open("data/raw/rtlcoder.jsonl", "w") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        response = row["Response"]
        code = response[0] if isinstance(response, list) else response
        fout.write(json.dumps({"instruction": row["Instruction"], "code": code}) + "\n")
os.remove("data/raw/_resyn27k.jsonl")

# MG-Verilog: "GaTech-EIC/MG-Verilog" is stored on the Hub as a raw
# datasets.save_to_disk() dump (merged_dataset/*.arrow + dataset_info.json),
# not a standard load_dataset()-loadable dataset -- load_dataset() on it
# either errors or silently returns the library's internal bookkeeping
# fields instead of real columns. Fetch the actual folder and load it with
# load_from_disk() instead. Real schema (from dataset_info.json, verified
# against the live repo): top-level "code" (str) and a NESTED "description"
# dict with sub-fields block_summary / detailed_global_summary /
# high_level_global_summary -- not a flat "description_detailed" field.
#
# Every description field also ships wrapped in a hardcoded Llama chat
# template ("<s>[INST] <<SYS>> You only complete chats with syntax correct
# Verilog code... <</SYS>>"). Feeding those literal special-token strings
# through OUR OWN prompt template would just train the model to emit stray
# "<s>[INST]" artifacts -- the opposite of the industry-clean-output goal
# -- so strip everything through the first "<</SYS>>" and keep the actual
# description that follows.
!pip install -q datasets huggingface_hub
from datasets import load_from_disk
from huggingface_hub import snapshot_download

mg_path = snapshot_download(repo_id="GaTech-EIC/MG-Verilog", repo_type="dataset")
mg_verilog = load_from_disk(f"{mg_path}/merged_dataset")

def clean_mg_instruction(text: str) -> str:
    return re.sub(r"^.*?<</SYS>>\s*", "", text, flags=re.DOTALL).strip()

with open('data/raw/mg_verilog.jsonl', 'w') as f:
    for row in mg_verilog:
        f.write(json.dumps({
            "instruction": clean_mg_instruction(row["description"]["detailed_global_summary"]),
            "code": row["code"],
        }) + "\n")

print('RTLCoder:', sum(1 for _ in open('data/raw/rtlcoder.jsonl')))
print('MG-Verilog:', sum(1 for _ in open('data/raw/mg_verilog.jsonl')))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

RTLCoder: 26532
MG-Verilog: 11144


In [7]:
# Eval sets -- EVAL ONLY, never in the training corpus. VerilogEval v2 (156
# problems) and RTLLM v2 (50 designs). Each row needs: id, instruction,
# testbench, top_module, tier (assigned via the tagger against the
# reference solution), code (reference solution -- used only for the
# contamination check, never for scoring).
!git clone --depth 1 https://github.com/NVlabs/verilog-eval /tmp/verilogeval
!git clone --depth 1 https://github.com/hkust-zhiyao/RTLLM /tmp/rtllm
# scripts/build_eval_jsonl.py's GLOB_PATTERNS must match whatever tag you
# just cloned -- both repos have reorganised their layout before, so fix
# the patterns there first if this step reports 0 problem directories.
!python -m scripts.build_eval_jsonl --src /tmp/verilogeval --out data/eval/verilogeval_v2.jsonl --benchmark verilogeval
!python -m scripts.build_eval_jsonl --src /tmp/rtllm --out data/eval/rtllm_v2.jsonl --benchmark rtllm

Cloning into '/tmp/verilogeval'...
remote: Enumerating objects: 796, done.
remote: Counting objects: 100% (796/796), done.
remote: Compressing objects: 100% (611/611), done.
remote: Total 796 (delta 296), reused 606 (delta 181), pack-reused 0 (from 0)
Receiving objects: 100% (796/796), 255.69 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (296/296), done.
Cloning into '/tmp/rtllm'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (592/592), done.
remote: Compressing objects: 100% (429/429), done.
remote: Total 592 (delta 188), reused 459 (delta 159), pack-reused 0 (from 0)
Receiving objects: 100% (592/592), 2.51 MiB | 18.32 MiB/s, done.
Resolving deltas: 100% (188/188), done.
[build_eval_jsonl] wrote 156 problems -> data/eval/verilogeval_v2.jsonl
[build_eval_jsonl] wrote 50 problems -> data/eval/rtllm_v2.jsonl


In [8]:
!python -m src.data.build_corpus \
  --sources rtlcoder=data/raw/rtlcoder.jsonl mg-verilog=data/raw/mg_verilog.jsonl \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --out artifacts/corpus.jsonl \
  --probe-frac 0.10 --seed 1337

[build_corpus] rtlcoder: 26532 raw -> 26532 cumulative valid rows
[build_corpus] mg-verilog: 11144 raw -> 37676 cumulative valid rows
[build_corpus] near-dup removal: dropped 2140, kept 35536
[build_corpus] eval-contamination removal: dropped 5, kept 35531
[build_corpus] tagging backend: 25356 via pyverilog AST, 10175 via regex fallback (28.6% fallback rate)
[build_corpus] split: 31978 train / 3553 probe
[build_corpus] wrote 35531 rows -> artifacts/corpus.jsonl


In [9]:
# Sanity-check tier/split balance before training on it
import json
from collections import Counter
rows = [json.loads(l) for l in open('artifacts/corpus.jsonl')]
print('total:', len(rows))
print('by split:', Counter(r['split'] for r in rows))
print('by tier (train):', Counter(r['tags']['tier'] for r in rows if r['split']=='train'))
print('by tier (probe):', Counter(r['tags']['tier'] for r in rows if r['split']=='probe'))

total: 35531
by split: Counter({'train': 31978, 'probe': 3553})
by tier (train): Counter({'T2': 12825, 'T4': 9498, 'T1': 8847, 'T3': 808})
by tier (probe): Counter({'T2': 1425, 'T4': 1055, 'T1': 983, 'T3': 90})


In [10]:
# Persist corpus + eval sets to Drive so 02_train/03_diagnose/04_eval can
# restore them even if they land on a different (fresh) Colab VM -- see
# the "restore from Drive" block in this notebook's bootstrap cell.
import shutil
for rel_path in ['artifacts/corpus.jsonl', 'data/eval/verilogeval_v2.jsonl', 'data/eval/rtllm_v2.jsonl']:
    shutil.copy(rel_path, f"{DATA}/{rel_path.split('/')[-1]}")
print('saved corpus.jsonl + eval-set jsonls to', DATA)

saved corpus.jsonl + eval-set jsonls to /content/drive/MyDrive/verilog-slm/data
